# 收益模型：联合拟合、预测与更新

本单元接收已构造的时间正序收益观测，支持 ARMA-GARCH、ARMA-EGARCH、ARMA-GJR-GARCH、ARMA-FIGARCH、ARMA-Component GARCH、ARFIMA-GARCH 六组。采样与去季节由 b03、c01、c02 完成；模型不重新采样或切日，也不在内部取 log、做整数差分或指数还原输入。

## 模型定义与估计目标

均值采用 $\Phi(L)(1-L)^d(r_t-\mu)=\Theta(L)\epsilon_t$，其中 $\Phi(L)=1-\sum\phi_iL^i$，$\Theta(L)=1+\sum\theta_jL^j$。ARMA 的 d 为 0；ARFIMA 的 d 与其他参数联合估计，限制于 (-0.5,0.5)，调用者可指定更窄区间。mean.mu 是该表示中的中心水平，不是 AR 方程截距。include_mean=False 时固定为 0。

全训练序列使用一次高斯准最大似然联合估计：

$$\mathcal L=\frac12\sum_t\left[\log(2\pi)+\log h_t+\epsilon_t^2/h_t\right].$$

每次候选参数都重新计算均值残差和方差递推。函数不先分别拟合两个模型再组合，不执行自动选阶；调用者选择模型和阶数，也可在训练范围内自行选择后调用相同函数。估计准则参考 [arch 正态似然定义](https://arch.readthedocs.io/en/latest/univariate/generated/generated/arch.univariate.Normal.loglikelihood.html)，联合均值表示及 Component 定义参考 [rugarch 方法说明](https://stat.ethz.ch/CRAN/web/packages/rugarch/vignettes/Introduction_to_the_rugarch_package.pdf)，平稳系数变换使用 [statsmodels Monahan 变换](https://www.statsmodels.org/stable/generated/statsmodels.tsa.statespace.tools.constrain_stationary_univariate.html)。

## 方差形式与约束

- GARCH：$h_t=\omega+\sum\alpha_i\epsilon_{t-i}^2+\sum\beta_jh_{t-j}$。系数非负，持久性严格小于 1。
- GJR：增加 $\sum\gamma_i\epsilon_{t-i}^2I(\epsilon_{t-i}<0)$。允许有效的负 gamma；分别保证正、负冲击系数非负，并限制对称 QML 下的持久性 $\sum\alpha+\frac12\sum\gamma+\sum\beta<1$。
- EGARCH：$\log h_t=\omega+\sum\alpha_i(|z_{t-i}|-\sqrt{2/\pi})+\sum\gamma_iz_{t-i}+\sum\beta_j\log h_{t-j}$。log 方差 AR 部分要求稳定，alpha、gamma 可正可负。
- FIGARCH：采用截断 ARCH 无穷表示。p、q 各支持 0 或 1；$0<d<1$，$0\le\phi\le(1-d)/2$，$0\le\beta\le d+\phi$。使用其专用正值条件，不将普通 GARCH 的持久性条件套在 FIGARCH 上。具体截断长度由 fractional_memory 指定。[arch FIGARCH 定义](https://arch.readthedocs.io/en/latest/univariate/generated/arch.univariate.FIGARCH.html)
- Component：$q_t=\omega+\rho q_{t-1}+\eta(\epsilon_{t-1}^2-h_{t-1})$，$h_t=q_t+\sum\alpha_i(\epsilon_{t-i}^2-q_{t-i})+\sum\beta_j(h_{t-j}-q_{t-j})$。长期递推为 Engle–Lee 一阶形式；短期 ARCH／GARCH 阶数由调用者指定，均至少为 1。采用非负冲击系数、$\sum\alpha+\sum\beta<\rho<1$ 与 $0<\eta<\beta_1$；实际路径中的 h、q 还须严格为正，否则明确失败。短期偏离 $h_t-q_t$ 可以为负，三个量满足恒等关系。

AR 与 MA 使用平稳／可逆参数化。ARFIMA 的分数滤波、FIGARCH 的历史核均使用调用者指定的有限截断；分数模型不是通过单独 Whittle 拟合再接 GARCH。所有模型的预测只针对本序列的下一采样步。

In [ ]:
from __future__ import annotations

from copy import deepcopy
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import expit, softmax
from statsmodels.tsa.statespace.tools import constrain_stationary_univariate, unconstrain_stationary_univariate
from arch.univariate import GARCH
from numba import njit

RETURN_MODEL_NAMES = (
    "ARMA-GARCH", "ARMA-EGARCH", "ARMA-GJR-GARCH", "ARMA-FIGARCH",
    "ARMA-Component GARCH", "ARFIMA-GARCH",
)

In [ ]:
@njit(cache=False)
def _fractional_return_weights(d, memory):
    weights = np.empty(memory + 1)
    weights[0] = 1.0
    for lag in range(1, memory + 1):
        weights[lag] = weights[lag - 1] * (lag - 1 - d) / lag
    return weights


def _decode_return_parameters(raw_parameters, specification):
    """从优化坐标得到满足对应约束的模型系数和实际递推核。"""
    ar_order, ma_order = specification["mean_order"]
    arch_order, garch_order = specification["volatility_order"]
    asymmetric_order = specification["asymmetric_order"]
    kind = specification["variance_kind"]
    parameters = {}
    cursor = 0
    mu = 0.0
    if specification["include_mean"]:
        mu = float(raw_parameters[cursor]); cursor += 1
        parameters["mean.mu"] = mu
    ar = (constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + ar_order]))
          if ar_order else np.empty(0))
    cursor += ar_order
    ma = (-constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + ma_order]))
          if ma_order else np.empty(0))
    cursor += ma_order
    for lag, coefficient in enumerate(ar, 1):parameters[f"mean.ar[{lag}]"] = float(coefficient)
    for lag, coefficient in enumerate(ma, 1):parameters[f"mean.ma[{lag}]"] = float(coefficient)
    mean_d = 0.0
    if specification["model"] == "ARFIMA-GARCH":
        low, high = specification["fractional_d_bounds"]
        mean_d = low + (high - low) * expit(raw_parameters[cursor]); cursor += 1
        parameters["mean.d"] = float(mean_d)
    mean_weights = (_fractional_return_weights(mean_d, specification["fractional_memory"])
                    if specification["model"] == "ARFIMA-GARCH" else np.ones(1))
    mean_filter = np.convolve(np.r_[1.0, -ar], mean_weights)
    omega = float(raw_parameters[cursor] if kind == 1 else np.exp(raw_parameters[cursor]))
    cursor += 1
    parameters["variance.omega"] = omega
    alpha, gamma, beta = np.empty(0), np.empty(0), np.empty(0)
    figarch_weights = np.empty(0)
    rho, eta = 0.0, 0.0
    if kind in (0, 2):
        size = arch_order + asymmetric_order + garch_order
        weights = softmax(np.r_[raw_parameters[cursor:cursor + size], 0.0])[:-1]
        alpha = weights[:arch_order].copy()
        for lag in range(min(arch_order, asymmetric_order)):alpha[lag] *= 2
        gamma = 2 * weights[arch_order:arch_order + asymmetric_order]
        for lag in range(min(arch_order, asymmetric_order)):gamma[lag] -= alpha[lag]
        beta = weights[arch_order + asymmetric_order:].copy()
    elif kind == 1:
        alpha = raw_parameters[cursor:cursor + arch_order].copy(); cursor += arch_order
        gamma = raw_parameters[cursor:cursor + asymmetric_order].copy(); cursor += asymmetric_order
        beta = (constrain_stationary_univariate(np.sinh(raw_parameters[cursor:cursor + garch_order]))
                if garch_order else np.empty(0))
    elif kind == 3:
        low, high = specification["fractional_d_bounds"]
        d = low + (high - low) * expit(raw_parameters[cursor]); cursor += 1
        phi = 0.5 * (1 - d) * expit(raw_parameters[cursor]) if arch_order else 0.0
        cursor += arch_order
        beta_value = (d + phi) * expit(raw_parameters[cursor]) if garch_order else 0.0
        parameters["variance.d"] = float(d)
        if arch_order:parameters["variance.phi"] = float(phi)
        beta = np.array([beta_value]) if garch_order else np.empty(0)
        fractional_weights = _fractional_return_weights(d, specification["fractional_memory"])
        figarch_weights = -lfilter([1.0, -phi], [1.0, -beta_value], fractional_weights)[1:]
    else:
        rho = float(expit(raw_parameters[cursor])); cursor += 1
        eta_fraction = expit(raw_parameters[cursor]); cursor += 1
        weights = rho * softmax(np.r_[raw_parameters[cursor:cursor + arch_order + garch_order], 0.0])[:-1]
        alpha, beta = weights[:arch_order].copy(), weights[arch_order:].copy()
        eta = float(beta[0] * eta_fraction)
        parameters["variance.rho"], parameters["variance.eta"] = rho, eta
    for name, coefficients in (("alpha", alpha), ("gamma", gamma), ("beta", beta)):
        for lag, coefficient in enumerate(coefficients, 1):parameters[f"variance.{name}[{lag}]"] = float(coefficient)
    return {"parameters": parameters, "mu": mu, "mean_filter": mean_filter, "ma": ma,
            "omega": omega, "alpha": alpha, "gamma": gamma, "beta": beta,
            "rho": rho, "eta": eta, "figarch_weights": figarch_weights}

In [ ]:
@njit(cache=False)
def _return_lag(values, presample, position, lag, default):
    source = position - lag
    if source >= 0:return values[source]
    source += len(presample)
    return presample[source] if source >= 0 else default


@njit(cache=False)
def _filter_return_sequence(
    returns, kind, mu, mean_filter, ma, omega, alpha, gamma, beta, rho, eta,
    figarch_weights, initial_variance, initial_component,
    past_returns, past_residuals, past_variances, past_components,
    initial_log_variance, past_log_variances,
):
    """训练、下一步预测和吸收观测共享同一均值／方差递推。"""
    size = len(returns)
    means, residuals = np.full(size, np.nan), np.full(size, np.nan)
    variances, components = np.full(size, np.nan), np.full(size, np.nan)
    log_variances = np.full(size, np.nan)
    log_max = np.log(np.finfo(np.float64).max)
    log_min = np.log(np.nextafter(0.0, 1.0))
    for position in range(size):
        conditional_mean = mu
        for lag in range(1, len(mean_filter)):
            conditional_mean -= mean_filter[lag] * (_return_lag(returns, past_returns, position, lag, mu) - mu)
        for lag in range(1, len(ma) + 1):
            conditional_mean += ma[lag - 1] * _return_lag(residuals, past_residuals, position, lag, 0.0)
        means[position] = conditional_mean
        variance = omega
        component = np.nan
        if kind in (0, 2):
            for lag in range(1, len(alpha) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else previous_variance
                variance += alpha[lag - 1] * squared_innovation
            for lag in range(1, len(gamma) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                negative_shock = innovation * innovation * (innovation < 0) if available else 0.5 * previous_variance
                variance += gamma[lag - 1] * negative_shock
            for lag in range(1, len(beta) + 1):
                variance += beta[lag - 1] * _return_lag(variances, past_variances, position, lag, initial_variance)
        elif kind == 1:
            log_variance = omega
            for lag in range(1, max(len(alpha), len(gamma)) + 1):
                available = position >= lag or len(past_residuals) >= lag - position
                if not available:continue
                previous_log_variance = _return_lag(log_variances, past_log_variances, position, lag, initial_log_variance)
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                # 直接在对数域标准化，避免先构造巨大／极小方差再开方。
                standardized = 0.0 if innovation == 0.0 else np.sign(innovation) * np.exp(np.log(abs(innovation)) - 0.5 * previous_log_variance)
                if lag <= len(alpha):log_variance += alpha[lag - 1] * (abs(standardized) - np.sqrt(2 / np.pi))
                if lag <= len(gamma):log_variance += gamma[lag - 1] * standardized
            for lag in range(1, len(beta) + 1):
                log_variance += beta[lag - 1] * _return_lag(log_variances, past_log_variances, position, lag, initial_log_variance)
            log_variances[position] = log_variance
            # 只表示可表示的原尺度值；不裁剪，QML 仍可使用有限 log(h)。
            variance = np.exp(log_variance) if log_min <= log_variance <= log_max else np.nan
        elif kind == 3:
            beta_value = beta[0] if len(beta) else 0.0
            variance = omega / (1 - beta_value)
            for lag in range(1, len(figarch_weights) + 1):
                previous_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else previous_variance
                variance += figarch_weights[lag - 1] * squared_innovation
        else:
            previous_variance = _return_lag(variances, past_variances, position, 1, initial_variance)
            previous_component = _return_lag(components, past_components, position, 1, initial_component)
            available = position >= 1 or len(past_residuals) >= 1 - position
            innovation = _return_lag(residuals, past_residuals, position, 1, 0.0)
            squared_innovation = innovation * innovation if available else previous_variance
            component = omega + rho * previous_component + eta * (squared_innovation - previous_variance)
            components[position] = component
            variance = component
            for lag in range(1, len(alpha) + 1):
                past_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                past_component = _return_lag(components, past_components, position, lag, initial_component)
                available = position >= lag or len(past_residuals) >= lag - position
                innovation = _return_lag(residuals, past_residuals, position, lag, 0.0)
                squared_innovation = innovation * innovation if available else past_variance
                variance += alpha[lag - 1] * (squared_innovation - past_component)
            for lag in range(1, len(beta) + 1):
                past_variance = _return_lag(variances, past_variances, position, lag, initial_variance)
                past_component = _return_lag(components, past_components, position, lag, initial_component)
                variance += beta[lag - 1] * (past_variance - past_component)
        if kind == 1:
            if not np.isfinite(log_variances[position]):
                return means, residuals, variances, components, log_variances
        elif not np.isfinite(variance) or variance <= 0 or (kind == 4 and (not np.isfinite(component) or component <= 0)):
            return means, residuals, variances, components, log_variances
        else:
            log_variances[position] = np.log(variance)
        variances[position] = variance
        residuals[position] = returns[position] - conditional_mean
        if not np.isfinite(residuals[position]):return means, residuals, variances, components, log_variances
    return means, residuals, variances, components, log_variances


def _evaluate_return_sequence(returns, state, *, require_variance_output=True):
    """对数域用于 EGARCH 递推／QML；正式预测仍须有可表示的输入尺度方差。"""
    kind = state["specification"]["variance_kind"]
    initial_log_variance = state.get("initial_log_variance", np.log(state["initial_variance"]))
    past_log_variances = state.get("log_variances_history", np.log(state["variances_history"]))
    means, residuals, variances, components, log_variances = _filter_return_sequence(
        np.asarray(returns, dtype=float), kind,
        state["mu"], state["mean_filter"], state["ma"], state["omega"],
        state["alpha"], state["gamma"], state["beta"], state["rho"], state["eta"],
        state["figarch_weights"], state["initial_variance"], state["initial_component"],
        state["returns_history"], state["residuals_history"],
        state["variances_history"], state["components_history"],
        initial_log_variance, past_log_variances,
    )
    valid = np.isfinite(means) & np.isfinite(residuals) & np.isfinite(log_variances)
    if kind != 1 or require_variance_output:
        valid &= np.isfinite(variances) & (variances > 0)
    if kind == 4:valid &= np.isfinite(components) & (components > 0)
    if not valid.all():
        position = int(np.flatnonzero(~valid)[0])
        if kind == 1 and np.isfinite(log_variances[position]) and (
                not np.isfinite(variances[position]) or variances[position] <= 0):
            log_h = float(log_variances[position])
            detail = dict(sequence_position=position, log_variance=log_h,
                          log_float64_min=float(np.log(np.nextafter(0.0, 1.0))),
                          log_float64_max=float(np.log(np.finfo(np.float64).max)))
            return {"status": "failed",
                    "reason": f"序列位置 {position} 的 EGARCH 方差超出 float64 正有限表示范围；log(h)={log_h:.12g}",
                    "numerical_diagnostics": detail}
        return {"status": "failed", "reason": f"序列位置 {position} 的均值、对数方差或长期成分无效"}
    return {"status": "ok", "reason": "", "mean": means, "residual": residuals,
            "variance": variances, "log_variance": log_variances, "long_term_variance": components}


## 三个调用接口

`fit_return_model` 接收一个有限值的一维序列；Pandas Series 保留其索引，数组使用顺序索引。model、mean_order=(AR,MA)、volatility_order=(ARCH,GARCH) 必须由调用者提供；GJR／EGARCH 的 asymmetric_order 默认 1，也可显式指定，其他组为 0。ARFIMA／FIGARCH 必须提供正整数 fractional_memory；其他组不使用该参数。没有在函数内部删除缺失行、自动选阶或遍历组合。

初始化可选 backcast 或 sample_variance；initial_variance 可显式覆盖初始方差，Component 还可指定 initial_component。presample 可提供按时间正序的 returns、residuals、variances、components 历史数组；未提供的历史以均值水平、零均值创新期望和初始方差／长期成分初始化。残差缺历史时，方差递推使用相应历史方差作为平方创新期望，GJR 负冲击期望为其一半，EGARCH 中心化冲击期望为 0。初始化和所有拟合数据只来自本次提供的训练输入。

优化器内部仅用训练标准差作线性数值换单位；预测、残差、方差、长期成分和具名系数都还原到调用者的输入尺度。EGARCH 的方差截距按 log 方差平移关系还原。numeric_scale 可由调用者明确指定正数。这里不改变输入的 log／非 log 语义。

fit 返回 status、reason、parameters、specification、state、in_sample、optimization、loglikelihood、next_forecast。训练行全部进入 QML，没有自动丢弃起始段。state 保留下一步所需的有限历史，而非持续累积全部测试序列。

`predict_return_model(state)` 不改变状态；`update_return_model(state, observed_return)` 只吸收一个实现值，返回新 state，保持参数固定且不修改旧 state。返回的 variance_forecast 是采样收益创新的条件方差；RV／BPV 分钟贡献观测的直接建模归 c04。Component 同时返回长期成分和短期偏离。

配置不合法、样本不足、缺失、零变化、优化不收敛、数值或有效性失败均返回 failed 和明确原因。失败拟合不提供可调用 state；失败更新不推进旧状态。只进行一次优化调用，不自动换模型、填值或重试。
EGARCH 在内存状态中保存 `initial_log_variance` 和 `log_variances_history`，拟合、预测、更新沿用同一个对数递推。QML 使用 `log(h)` 和对数域计算的平方标准化残差，避免为了计算似然而先指数化巨大方差。正式输出仍须有输入尺度上可表示的正有限方差；超出范围返回 `failed`、包含 `log(h)` 的原因及 `numerical_diagnostics`，不裁剪、不将越界值改成最大浮点数，也不推进旧状态。

EGARCH 的模型定义见 [Nelson 原文](https://web.pdx.edu/~crkl/ec572/readings/Nelson91.pdf)。波动过程条件与观测驱动滤波的可逆条件需要区分，见 [Wintenberger 的连续可逆性研究](https://arxiv.org/html/1211.3292)。本函数保留既定参数范围，不据短样本的数值越界自动新增该文的估计约束；优化器报告收敛后仍检查训练路径及下一步预测的有效性。


In [ ]:
def fit_return_model(
    observations, *, model: str, mean_order: tuple[int, int],
    volatility_order: tuple[int, int], asymmetric_order: int | None = None,
    include_mean: bool = True, fractional_memory: int | None = None,
    fractional_d_bounds: tuple[float, float] | None = None,
    initialization: str = "backcast", initial_variance: float | None = None,
    initial_component: float | None = None, presample: dict | None = None,
    numeric_scale: float | None = None, optimizer_options: dict | None = None,
) -> dict:
    """对一个已选模型／阶数进行一次全序列 Gaussian QML 联合估计。"""
    failure = {"status": "failed", "reason": "", "parameters": {}, "state": None}
    try:
        if model not in RETURN_MODEL_NAMES:raise ValueError("model 不属于批准的六组收益模型")
        for name, orders in (("mean_order", mean_order), ("volatility_order", volatility_order)):
            if len(orders) != 2 or any(not isinstance(value, (int, np.integer)) or isinstance(value, (bool, np.bool_)) or value < 0 for value in orders):
                raise ValueError(f"{name} 必须为两个非负整数")
        ar_order, ma_order = tuple(map(int, mean_order))
        arch_order, garch_order = tuple(map(int, volatility_order))
        kind = {"ARMA-GARCH": 0, "ARFIMA-GARCH": 0, "ARMA-EGARCH": 1,
                "ARMA-GJR-GARCH": 2, "ARMA-FIGARCH": 3, "ARMA-Component GARCH": 4}[model]
        asymmetric_order = (1 if kind in (1, 2) else 0) if asymmetric_order is None else asymmetric_order
        if not isinstance(asymmetric_order, (int, np.integer)) or isinstance(asymmetric_order, (bool, np.bool_)) or asymmetric_order < 0:
            raise ValueError("asymmetric_order 必须为非负整数")
        if kind not in (1, 2) and asymmetric_order:raise ValueError("该模型不使用 asymmetric_order")
        if kind == 3 and (arch_order > 1 or garch_order > 1):raise ValueError("FIGARCH 的 p、q 目前各仅支持 0 或 1")
        if kind == 4 and (arch_order < 1 or garch_order < 1):raise ValueError("Component 短期 ARCH、GARCH 阶数均须至少为 1")
        if not isinstance(include_mean, (bool, np.bool_)):raise ValueError("include_mean 必须为布尔值")
        fractional = model == "ARFIMA-GARCH" or kind == 3
        if fractional:
            if not isinstance(fractional_memory, (int, np.integer)) or isinstance(fractional_memory, (bool, np.bool_)) or fractional_memory < 1:
                raise ValueError("分数模型必须显式提供正整数 fractional_memory")
            d_bounds = ((0.0, 1.0) if kind == 3 else (-0.5, 0.5)) if fractional_d_bounds is None else fractional_d_bounds
            low, high = map(float, d_bounds)
            domain_low, domain_high = (0.0, 1.0) if kind == 3 else (-0.5, 0.5)
            if not np.isfinite([low, high]).all() or not domain_low <= low < high <= domain_high:
                raise ValueError("fractional_d_bounds 须为模型允许范围内的递增有限边界")
            d_bounds = (low, high)
        else:
            if fractional_memory is not None or fractional_d_bounds is not None:
                raise ValueError("非分数模型不使用 fractional_memory 或 fractional_d_bounds")
            fractional_memory, d_bounds = 0, None
        if initialization not in ("backcast", "sample_variance"):raise ValueError("initialization 必须为 backcast 或 sample_variance")
        if kind != 4 and initial_component is not None:raise ValueError("只有 Component 使用 initial_component")
        if isinstance(observations, pd.Series):
            values = observations.to_numpy(dtype=float, na_value=np.nan)
            observation_index = observations.index.copy()
        else:
            values = np.asarray(observations, dtype=float)
            observation_index = pd.RangeIndex(len(values)) if values.ndim else pd.RangeIndex(0)
        if values.ndim != 1 or not len(values) or not np.isfinite(values).all():
            raise ValueError("observations 必须为非空有限一维序列；不自动删除缺失行")
        sample_scale = float(np.std(values))
        if not np.isfinite(sample_scale) or sample_scale <= 0:raise ValueError("训练收益没有有效变化，不能估计非退化随机模型")
        scale = sample_scale if numeric_scale is None else float(numeric_scale)
        if not np.isfinite(scale) or scale <= 0 or not np.isfinite(scale * scale) or scale * scale == 0:
            raise ValueError("numeric_scale 必须为可表示其平方的有限正数")
        scaled_returns = values / scale
        mu_start = float(np.mean(scaled_returns)) if include_mean else 0.0
        residual_seed = scaled_returns - mu_start
        if initial_variance is None:
            variance_start = (float(GARCH().backcast(residual_seed)) if initialization == "backcast"
                              else float(np.mean(residual_seed ** 2)))
        else:variance_start = float(initial_variance) / scale ** 2
        component_start = variance_start if initial_component is None else float(initial_component) / scale ** 2
        if not np.isfinite([variance_start, component_start]).all() or min(variance_start, component_start) <= 0:
            raise ValueError("初始方差与长期成分必须严格为正且有限")
        specification = {"model": model, "mean_order": (ar_order, ma_order),
            "volatility_order": (arch_order, garch_order), "asymmetric_order": int(asymmetric_order),
            "include_mean": bool(include_mean), "fractional_memory": int(fractional_memory),
            "fractional_d_bounds": d_bounds, "variance_kind": kind,
            "initialization": initialization, "numeric_scale": scale, "estimation": "gaussian_qml"}
        presample = {} if presample is None else presample
        if set(presample) - {"returns", "residuals", "variances", "components"}:raise ValueError("presample 存在未定义的历史字段")
        history = {}
        for name in ("returns", "residuals", "variances", "components"):
            past_values = np.asarray(presample.get(name, []), dtype=float)
            if past_values.ndim != 1 or not np.isfinite(past_values).all():raise ValueError(f"presample.{name} 必须为有限一维历史")
            if name in ("variances", "components") and np.any(past_values <= 0):raise ValueError(f"presample.{name} 必须严格为正")
            history[f"{name}_history"] = past_values / (scale ** 2 if name in ("variances", "components") else scale)
        raw_start, bounds = [], []
        if include_mean:raw_start.append(mu_start); bounds.append((None, None))
        raw_start.extend([0.0] * (ar_order + ma_order)); bounds.extend([(-8.0, 8.0)] * (ar_order + ma_order))
        if model == "ARFIMA-GARCH":
            d_start = 0.15 if d_bounds[0] < 0.15 < d_bounds[1] else sum(d_bounds) / 2
            proportion = (d_start - d_bounds[0]) / (d_bounds[1] - d_bounds[0])
            raw_start.append(float(np.log(proportion / (1 - proportion)))); bounds.append((-14.0, 14.0))
        omega_position = len(raw_start)
        raw_start.append(0.0 if kind == 1 else float(np.log(0.1 * variance_start)))
        bounds.append((None, None) if kind == 1 else (-30.0, 30.0))
        if kind in (0, 2):
            weights = np.r_[np.full(arch_order, 0.06 / arch_order) if arch_order else [],
                             np.full(asymmetric_order, 0.04 / asymmetric_order) if asymmetric_order else [],
                             np.full(garch_order, 0.84 / garch_order) if garch_order else []]
            raw_start.extend(np.log(weights / (1 - weights.sum())))
            bounds.extend([(-20.0, 20.0)] * len(weights))
        elif kind == 1:
            raw_start.extend([0.10] * arch_order + [-0.05] * asymmetric_order)
            bounds.extend([(None, None)] * (arch_order + asymmetric_order))
            if garch_order:
                stable_beta = np.full(garch_order, 0.85 / garch_order)
                raw_start.extend(np.arcsinh(unconstrain_stationary_univariate(stable_beta)))
                bounds.extend([(-8.0, 8.0)] * garch_order)
        elif kind == 3:
            d_start = 0.3 if d_bounds[0] < 0.3 < d_bounds[1] else sum(d_bounds) / 2
            proportion = (d_start - d_bounds[0]) / (d_bounds[1] - d_bounds[0])
            raw_start.append(float(np.log(proportion / (1 - proportion)))); bounds.append((-14.0, 14.0))
            if arch_order:raw_start.append(0.0); bounds.append((-20.0, 20.0))
            if garch_order:raw_start.append(float(np.log(0.6 / 0.4))); bounds.append((-20.0, 20.0))
        else:
            rho_start = 0.97
            raw_start.extend([np.log(rho_start / (1 - rho_start)), np.log(0.02 / 0.98)])
            bounds.extend([(-14.0, 14.0), (-20.0, 20.0)])
            weights = np.r_[np.full(arch_order, 0.08 / arch_order), np.full(garch_order, 0.83 / garch_order)] / rho_start
            raw_start.extend(np.log(weights / (1 - weights.sum())))
            bounds.extend([(-20.0, 20.0)] * len(weights))
        raw_start = np.asarray(raw_start, dtype=float)
        if len(values) <= max(len(raw_start), ar_order, ma_order, arch_order, garch_order, asymmetric_order):
            raise ValueError("训练观测数须超过自由参数数与各阶数")
        seed_coefficients = _decode_return_parameters(raw_start, specification)
        if kind in (0, 2):
            persistence = seed_coefficients["alpha"].sum() + 0.5 * seed_coefficients["gamma"].sum() + seed_coefficients["beta"].sum()
            omega_start = (1 - persistence) * variance_start
        elif kind == 1:omega_start = (1 - seed_coefficients["beta"].sum()) * np.log(variance_start)
        elif kind == 3:
            beta_value = seed_coefficients["beta"][0] if garch_order else 0.0
            omega_start = (1 - beta_value) * (1 - seed_coefficients["figarch_weights"].sum()) * variance_start
        else:omega_start = (1 - seed_coefficients["rho"]) * variance_start
        raw_start[omega_position] = omega_start if kind == 1 else np.log(max(omega_start, 1.e-12))
        seed_state = {"specification": specification, "initial_variance": variance_start,
                      "initial_component": component_start, **history}

        def objective(raw_parameters):
            candidate = {**seed_state, **_decode_return_parameters(raw_parameters, specification)}
            path = _evaluate_return_sequence(scaled_returns, candidate, require_variance_output=False)
            if path["status"] != "ok":return 1.e12
            with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
                if kind == 1:
                    squared_standardized = np.zeros(len(scaled_returns))
                    nonzero = path["residual"] != 0
                    squared_standardized[nonzero] = np.exp(
                        2 * np.log(np.abs(path["residual"][nonzero])) - path["log_variance"][nonzero])
                    score = 0.5 * np.mean(np.log(2 * np.pi) + path["log_variance"] + squared_standardized)
                else:
                    score = 0.5 * np.mean(np.log(2 * np.pi) + np.log(path["variance"]) + path["residual"] ** 2 / path["variance"])
            return float(score) if np.isfinite(score) else 1.e12

        initial_objective = objective(raw_start)
        options = {"maxiter": 500, "maxfun": 100000, "ftol": 1.e-10, "gtol": 1.e-5, "maxls": 40}
        if optimizer_options is not None:options.update(optimizer_options)
        optimization = minimize(objective, raw_start, method="L-BFGS-B", bounds=bounds, options=options)
        diagnostics = {"method": "L-BFGS-B", "success": bool(optimization.success),
            "message": str(optimization.message), "iterations": int(optimization.nit),
            "evaluations": int(optimization.nfev), "initial_average_qml": initial_objective,
            "final_average_qml": float(optimization.fun), "joint_parameters": len(raw_start)}
        if not optimization.success:
            return {**failure, "reason": f"联合优化未收敛: {optimization.message}", "optimization": diagnostics,
                    "specification": specification}
        coefficients = _decode_return_parameters(optimization.x, specification)
        # 线性换单位仅用于数值优化；所有正式系数与状态回到输入尺度。
        coefficients["mu"] *= scale
        if include_mean:coefficients["parameters"]["mean.mu"] = coefficients["mu"]
        if kind == 1:coefficients["omega"] += (1 - coefficients["beta"].sum()) * 2 * np.log(scale)
        else:coefficients["omega"] *= scale ** 2
        coefficients["parameters"]["variance.omega"] = coefficients["omega"]
        initial_state = {"status": "ok", "specification": specification, **coefficients,
            "initial_variance": variance_start * scale ** 2, "initial_component": component_start * scale ** 2,
            **{name: np.asarray(presample.get(name.removesuffix("_history"), []), dtype=float).copy() for name in history}}
        if kind == 1:
            initial_state["initial_log_variance"] = float(np.log(initial_state["initial_variance"]))
            initial_state["log_variances_history"] = np.log(initial_state["variances_history"])
        path = _evaluate_return_sequence(values, initial_state)
        if path["status"] != "ok":return {**failure, "reason": path["reason"], "optimization": diagnostics,
                                          "numerical_diagnostics": path.get("numerical_diagnostics")}
        history_limit = max(len(coefficients["mean_filter"]) - 1, ma_order, arch_order, garch_order,
                            asymmetric_order, fractional_memory if kind == 3 else 0, 1)
        state = deepcopy(initial_state)
        for name, realized_values in (("returns", values), ("residuals", path["residual"]), ("variances", path["variance"])):
            state[f"{name}_history"] = np.r_[initial_state[f"{name}_history"], realized_values][-history_limit:].copy()
        state["components_history"] = (np.r_[initial_state["components_history"], path["long_term_variance"]][-history_limit:].copy()
                                       if kind == 4 else np.empty(0))
        if kind == 1:
            state["log_variances_history"] = np.r_[initial_state["log_variances_history"], path["log_variance"]][-history_limit:].copy()
        state["history_limit"], state["observations_seen"] = history_limit, len(values)
        forecast = predict_return_model(state)
        if forecast["status"] != "ok":return {**failure, "reason": forecast["reason"], "optimization": diagnostics,
                                              "numerical_diagnostics": forecast.get("numerical_diagnostics")}
        with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
            standardized_innovations = path["residual"] / np.sqrt(path["variance"])
            loglikelihood = -float(0.5 * np.sum(np.log(2 * np.pi) + np.log(path["variance"]) + standardized_innovations ** 2))
        if not np.isfinite(standardized_innovations).all() or not np.isfinite(loglikelihood):
            return {**failure, "reason": "标准化创新或原尺度 QML 值非有限", "optimization": diagnostics}
        ar_values = [coefficients["parameters"][f"mean.ar[{lag}]"] for lag in range(1, ar_order + 1)]
        for polynomial in (np.r_[1.0, -np.asarray(ar_values)], np.r_[1.0, coefficients["ma"]],
                           np.r_[1.0, -coefficients["beta"]] if kind == 1 else np.ones(1)):
            if np.any(np.abs(np.polynomial.polynomial.polyroots(polynomial)) <= 1):
                return {**failure, "reason": "拟合系数的数值根不在平稳／可逆范围", "optimization": diagnostics}
        in_sample_df = pd.DataFrame({"observation": values, "conditional_mean": path["mean"],
            "innovation": path["residual"], "conditional_variance": path["variance"],
            "standardized_innovation": standardized_innovations}, index=observation_index)
        if kind == 4:
            in_sample_df["long_term_variance"] = path["long_term_variance"]
            in_sample_df["short_term_deviation"] = path["variance"] - path["long_term_variance"]
        return {"status": "ok", "reason": "", "parameters": coefficients["parameters"].copy(),
                "specification": specification.copy(), "state": state, "in_sample": in_sample_df,
                "optimization": diagnostics, "loglikelihood": loglikelihood, "next_forecast": forecast}
    except (ValueError, TypeError, FloatingPointError, OverflowError) as error:
        return {**failure, "reason": str(error)}


def predict_return_model(state: dict) -> dict:
    """下一采样步的条件均值／创新方差；预测不改变状态。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "预测需要成功拟合或更新后的有效 state"}
    path = _evaluate_return_sequence(np.array([0.0]), state)
    if path["status"] != "ok":return path
    forecast = {"status": "ok", "reason": "", "mean_forecast": float(path["mean"][0]),
                "variance_forecast": float(path["variance"][0])}
    if state["specification"]["variance_kind"] == 4:
        forecast["long_term_variance_forecast"] = float(path["long_term_variance"][0])
        forecast["short_term_deviation_forecast"] = forecast["variance_forecast"] - forecast["long_term_variance_forecast"]
    return forecast


def update_return_model(state: dict, observed_return: float) -> dict:
    """吸收一个实现收益，返回新状态；不重新估计系数，也不修改旧状态。"""
    if not isinstance(state, dict) or state.get("status") != "ok":
        return {"status": "failed", "reason": "更新需要成功拟合或更新后的有效 state", "state": None}
    try:observed_return = float(observed_return)
    except (ValueError, TypeError):
        return {"status": "failed", "reason": "实现收益必须为一个有限数值", "state": None}
    if not np.isfinite(observed_return):
        return {"status": "failed", "reason": "实现收益缺失或非有限；旧状态未推进", "state": None}
    path = _evaluate_return_sequence(np.array([observed_return]), state)
    if path["status"] != "ok":return {**path, "state": None}
    updated_state = deepcopy(state)
    for name, value in (("returns", observed_return), ("residuals", path["residual"][0]), ("variances", path["variance"][0])):
        updated_state[f"{name}_history"] = np.r_[state[f"{name}_history"], value][-state["history_limit"]:]
    if state["specification"]["variance_kind"] == 1:
        past_logs = state.get("log_variances_history", np.log(state["variances_history"]))
        updated_state["log_variances_history"] = np.r_[past_logs, path["log_variance"][0]][-state["history_limit"]:]
    if state["specification"]["variance_kind"] == 4:
        updated_state["components_history"] = np.r_[state["components_history"], path["long_term_variance"][0]][-state["history_limit"]:]
    updated_state["observations_seen"] += 1
    next_forecast = predict_return_model(updated_state)
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        standardized_innovation = float(path["residual"][0] / np.sqrt(path["variance"][0]))
    if next_forecast["status"] != "ok" or not np.isfinite(standardized_innovation):
        reason = next_forecast["reason"] or "标准化创新非有限"
        return {"status": "failed", "reason": f"吸收观测后下一步无法有效预测: {reason}", "state": None,
                "numerical_diagnostics": next_forecast.get("numerical_diagnostics")}
    output = {"status": "ok", "reason": "", "state": updated_state, "next_forecast": next_forecast,
        "conditional_mean": float(path["mean"][0]), "conditional_variance": float(path["variance"][0]),
        "innovation": float(path["residual"][0]),
        "standardized_innovation": standardized_innovation}
    if state["specification"]["variance_kind"] == 4:
        output["long_term_variance"] = float(path["long_term_variance"][0])
        output["short_term_deviation"] = output["conditional_variance"] - output["long_term_variance"]
    return output

In [ ]:
RETURN_MODEL_FIELD_LABELS = {
    "model": "模型", "status": "状态", "reason": "原因", "iterations": "联合优化迭代数",
    "mean_forecast": "下一步收益均值", "variance_forecast": "下一步条件方差",
    "long_term_variance_forecast": "下一步长期方差成分",
    "short_term_deviation_forecast": "下一步短期偏离",
    "observation": "实现收益", "conditional_mean": "条件均值",
    "innovation": "创新残差", "conditional_variance": "条件方差",
    "standardized_innovation": "标准化创新", "long_term_variance": "长期方差成分",
    "short_term_deviation": "短期偏离", "parameter": "具名系数", "value": "系数值",
}
RETURN_MODEL_COLUMN_LABELS = {
    name: f"{name}（{label}）" for name, label in RETURN_MODEL_FIELD_LABELS.items()
}

In [ ]:
import pyarrow as pa
from copy import deepcopy
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import transform_model_input, initial_model_input_position

demo_prediction_fields = [
    pa.field("sample_id", pa.int64(), nullable=False), pa.field("sample_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False), pa.field("sample_side", pa.string(), nullable=False),
    pa.field("training_start_sample_id", pa.int64()), pa.field("training_end_sample_id", pa.int64()),
    pa.field("training_points", pa.int64(), nullable=False), pa.field("model", pa.string(), nullable=False),
    pa.field("status", pa.string(), nullable=False), pa.field("reason", pa.string(), nullable=False),
    pa.field("mean_forecast", pa.float64()), pa.field("parameters", pa.map_(pa.string(), pa.float64())),
    pa.field("update_status", pa.string()), pa.field("update_reason", pa.string()),
    pa.field("updated_mean_forecast", pa.float64()),
]

def _compute_return_demo(demo_observations, *, history_points=500,
                         models=RETURN_MODEL_NAMES, target_positions=None):
    """逐目标重新估计；分片只选择目标，不切片训练历史；返回内存验收结果。"""
    demo_history_points = history_points
    selected_positions = np.arange(len(demo_observations), dtype=int) if target_positions is None else np.asarray(target_positions, dtype=int)
    if selected_positions.ndim != 1 or len(np.unique(selected_positions)) != len(selected_positions):
        raise ValueError("目标坐标必须是一维不重复位置")
    if len(selected_positions) and (selected_positions.min() < 0 or selected_positions.max() >= len(demo_observations)):
        raise ValueError("目标坐标超出完整输入范围")
    assert demo_observations.iloc[0]["log_return_status"] == "no_previous_sample"
    demo_inputs = transform_model_input(demo_observations, observation_col="log_return")
    demo_values = demo_inputs["model_input"].to_numpy(dtype=float, na_value=np.nan)
    initial_position = initial_model_input_position(demo_inputs)
    demo_result_schema = pa.schema([*demo_prediction_fields,
        *[pa.field(name, pa.float64()) for name in ("variance_forecast", "long_term_variance_forecast",
          "short_term_deviation_forecast", "updated_variance_forecast")]])
    demo_results = {}
    demo_update_checks = {}
    for model_name in models:
        rows, update_checks = [], 0
        options = dict(model=model_name, mean_order=(1, 1), volatility_order=(1, 1),
            fractional_memory=30 if model_name in ("ARFIMA-GARCH", "ARMA-FIGARCH") else None)
        for completed, position in enumerate(selected_positions, 1):
            target = demo_inputs.iloc[position]
            start = max(initial_position, position - demo_history_points)
            history = demo_values[start:position]
            row = dict(sample_id=target.sample_id, sample_at=target.sample_at, trading_date=target.trading_date,
                sample_side=target.sample_side, training_start_sample_id=demo_inputs.iloc[start].sample_id if len(history) else None,
                training_end_sample_id=demo_inputs.iloc[position - 1].sample_id if len(history) else None,
                training_points=len(history), model=model_name, parameters=None)
            fit = fit_return_model(history, **options)
            row.update(status=fit["status"], reason=fit["reason"])
            if fit["status"] == "ok":
                original_state = deepcopy(fit["state"])
                prediction = predict_return_model(fit["state"])
                row.update(status=prediction["status"], reason=prediction["reason"], parameters=fit["parameters"])
                if prediction["status"] == "ok":
                    row.update({k: v for k, v in prediction.items() if k not in ("status", "reason")})
                    np.testing.assert_allclose(prediction["mean_forecast"], fit["next_forecast"]["mean_forecast"])
                    np.testing.assert_allclose(prediction["variance_forecast"], fit["next_forecast"]["variance_forecast"])
                    update = update_return_model(fit["state"], target.model_input)
                    row.update(update_status=update["status"], update_reason=update["reason"])
                    assert fit["state"]["observations_seen"] == original_state["observations_seen"]
                    for key in ("returns_history", "residuals_history", "variances_history"):
                        np.testing.assert_array_equal(fit["state"][key], original_state[key])
                    if update["status"] == "ok":
                        assert update["state"]["parameters"] == fit["parameters"]
                        assert update["state"]["observations_seen"] == len(history) + 1
                        np.testing.assert_allclose(update["conditional_mean"], prediction["mean_forecast"])
                        np.testing.assert_allclose(update["conditional_variance"], prediction["variance_forecast"])
                        row.update(updated_mean_forecast=update["next_forecast"]["mean_forecast"],
                            updated_variance_forecast=update["next_forecast"]["variance_forecast"])
                        update_checks += 1
                    if model_name == "ARMA-Component GARCH":
                        np.testing.assert_allclose(row["variance_forecast"], row["long_term_variance_forecast"] + row["short_term_deviation_forecast"])
            rows.append(row)
            if completed % 100 == 0 or completed == len(selected_positions):
                print(f"c03 {model_name} {completed}/{len(selected_positions)} 点；成功更新检查 {update_checks}", flush=True)
        result = pd.DataFrame(rows).reindex(columns=demo_result_schema.names)
        assert len(result) == len(selected_positions)
        assert result.sample_id.tolist() == demo_inputs.iloc[selected_positions].sample_id.tolist()
        assert result.loc[result.training_end_sample_id.notna(), "training_end_sample_id"].lt(
            result.loc[result.training_end_sample_id.notna(), "sample_id"]).all()
        demo_results[model_name] = result
        demo_update_checks[model_name] = update_checks
    return dict(results=demo_results, update_checks=demo_update_checks, schema=demo_result_schema)


## Demo：完整收益序列逐点重估

用途：读取 c02 不去季节观测，调用六组正式收益模型的联合 QML 拟合、预测及状态更新。每个目标使用完整序列中此前最多 500 个采样点重新估计；初始期使用现有历史，失败位置和原因保留。输出含下一步均值／方差预测、具名系数、Component 成分及更新检查。

完整范围为 RB 2010-01-01—2026-09-30，训练截至 2024-12-31，测试自 2025-01-01，N15 GK sigma × 全合约成交额，预期 78,007 点。阶数为 (1,1)，分数历史截断 30 点。本单元与 c04 共用一个 models 作业、14 份产物和完成记录；两个 Notebook 执行同一配置时接续同一 UUID。节点和两个 CPU 只分配目标点，不截断完整输入中的训练历史。
上游完整 demo 未就绪时，试错读取全量真实分钟与既定采样位置，再切小样本，调用已有上游函数在内存准备输入。试错不消费来源不匹配的旧成果，不保存结果。正式 DLC 仍须使用已验收的完整上游成果；补齐后重新执行读取输入单元。


## 试错 demo

完整输入只读取一次；以下试错调用正式函数，仅在内存检查和展示。

In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
    _demo_products, _demo_open_run, _demo_upload, _demo_submit, _demo_wait, _demo_recover, _demo_read_saved,
    _read_real_demo, _load_intraday_demo_inputs, _compute_intraday_demo,
    _demo_load_positions, _demo_read_trial_sources, _demo_trial_prefix)
demo_stage = "models"
full_inputs, trial_sources, full_input_issue = None, None, ""
observation_path = candidate_root / list(_demo_products("observations").values())[0]
try:
    observations_df, observation_identity = _read_real_demo(observation_path)
    full_inputs = dict(frames={"observations": observations_df}, scope=observation_identity["scope"],
                       upstream={"observations": observation_identity})
except FileNotFoundError as error:
    full_input_issue = str(error)
except RuntimeError as error:
    if not ("计算定义已变化，请重算" in str(error) or "已变化，请重算" in str(error)):
        raise
    full_input_issue = str(error)
if full_inputs is None:
    print("c02 完整成果尚未就绪；先用当前上游函数在内存准备小样本。", full_input_issue)
    trial_sources = _demo_read_trial_sources(candidate_root)
else:
    print(f'完整既定采样位置：{len(full_inputs["frames"]["observations"]):,} 点')


In [ ]:
if full_inputs is None:
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c02_ModelObservations import (
        prepare_minute_observations, aggregate_model_observations)
    trial_prefix = _demo_trial_prefix(trial_sources, include_seasonality=False)
    trial_minute_df = prepare_minute_observations(trial_prefix["computed"]["contributions"])
    trial_observations_df = aggregate_model_observations(trial_minute_df, trial_prefix["positions"])
else:
    trial_observations_df = full_inputs["frames"]["observations"].iloc[:64].copy()
trial_history_points = 30  # 试错规模；完整 demo 仍为 500。


In [ ]:
trial_computed = _compute_return_demo(trial_observations_df, history_points=trial_history_points)
trial_tables = trial_computed["results"]


In [ ]:
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
for configuration, trial_frame in trial_tables.items():
    print(configuration, "；试错结果仅在内存展示")
    selected_columns = [name for name in demo_display_labels if name in trial_frame.columns]
    display(trial_frame.loc[:, selected_columns].iloc[sorted(set(range(min(4, len(trial_frame)))) | set(range(max(0, len(trial_frame) - 4), len(trial_frame))))].rename(columns=demo_display_labels))
    if "status" in trial_frame:
        display(trial_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())

    display(trial_frame.groupby(["status", "reason"], dropna=False).size().rename("行数").rename_axis(["status（状态）", "reason（原因）"]).to_frame())
    display(trial_frame.groupby(["update_status", "update_reason"], dropna=False).size().rename("行数").rename_axis(["update_status（更新状态）", "update_reason（更新原因）"]).to_frame())


## 正式 demo（DLC）

按参数与输入身份复用本地完整成果或接续已有作业；没有才提交。云端只加载导出定义，跳过本地试错、提交、查询及展示。

In [ ]:
demo_parameters = dict(start="2010-01-01", end="2026-09-30", test_start="2025-01-01",
    sampling="N15_gk_money_all_contracts_log_interpolate_fixed", history_points=500,
    mean_order=[1, 1], volatility_order=[1, 1], fractional_memory=30,
    har_windows=[1, 5, 20], mem_order=[1, 1], transform="identity",
    seasonality=None, require_nonnegative_prediction=True, refit="every_point")
demo_resources = dict(region="cn-shenzhen", workspace_id="351454",
    oss_bucket="oss-pai-q00s96z5hnrhywm9ns-cn-shenzhen",
    image_uri="dsw-registry-vpc.cn-shenzhen.cr.aliyuncs.com/pai/python:3.13.9-cpu-ubuntu22.04-9e1c8c7e-1764329431",
    instance_type="ecs.c6.large", instance_count=10)
demo_recompute = False  # 默认复用／接续；仅明确重算且旧作业已确认终态时改为 True。


In [ ]:
if full_inputs is None:
    raise RuntimeError("完整上游成果尚未就绪。完成上游 DLC 后重新执行读取输入单元；内存试错不会用于完整任务。\n" + full_input_issue)

if demo_stage != "seasonality":
    reference = full_inputs["frames"]["positions" if demo_stage == "observations" else "observations"]
    if len(reference) != 78007:
        raise RuntimeError("本次完整 demo 应覆盖 78,007 个既定采样位置")
demo_context = _demo_open_run(candidate_root, demo_stage, full_inputs, demo_parameters,
                              demo_resources, recompute=demo_recompute)
demo_recompute = False
print("本地完整成果有效，跳过云计算" if demo_context["cached"] else
      f'{demo_context["record"]["name"]} | {demo_context["record"]["state"]}')


In [ ]:
def _model_demo_cpu_slice(code_root, input_dir, rank, node_count, cpu_slot):
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import _demo_node_inputs
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c04_VolatilityModels import _compute_volatility_demo
    plan, frames = _demo_node_inputs(code_root, input_dir, frame_names=("observations",))
    observations = frames["observations"]
    # 两 CPU 分配独立目标，各目标仍从完整序列取此前历史。
    targets = np.arange(rank + cpu_slot * node_count, len(observations), node_count * 2, dtype=int)
    history = plan["identity"]["parameters"]["history_points"]
    return dict(returns=_compute_return_demo(observations, history_points=history, target_positions=targets),
                volatility=_compute_volatility_demo(observations, history_points=history, target_positions=targets))


def run_models_demo_node(code_root, input_dir, output_dir, *, rank, node_count):
    from concurrent.futures import ProcessPoolExecutor
    import multiprocessing
    from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import (
        _demo_node_inputs, _demo_save_node, _demo_stage_node_inputs, _demo_products)
    input_dir = _demo_stage_node_inputs(code_root, input_dir, rank)
    plan, _ = _demo_node_inputs(code_root, input_dir, frame_names=())
    if node_count != 10 or node_count != plan["identity"]["resources"]["instance_count"] or not 0 <= rank < node_count:
        raise ValueError("节点身份不匹配")
    with ProcessPoolExecutor(max_workers=2, mp_context=multiprocessing.get_context("spawn")) as pool:
        parts = [future.result() for future in
                 [pool.submit(_model_demo_cpu_slice, code_root, input_dir, rank, node_count, slot) for slot in range(2)]]
    history = plan["identity"]["parameters"]["history_points"]
    partition = dict(rank=rank, node_count=node_count, selection="position_modulo",
                     total_target_points=plan["identity"]["frames"]["observations"]["rows"])
    upstream = plan["metadata"]["upstream"]
    products, names = {}, list(_demo_products("models"))
    for index, model in enumerate(RETURN_MODEL_NAMES):
        frame = pd.concat([p["returns"]["results"][model] for p in parts], ignore_index=True).sort_values("sample_id")
        parameters = dict(model=model, mean_order=(1, 1), volatility_order=(1, 1),
            fractional_memory=30 if model in ("ARFIMA-GARCH", "ARMA-FIGARCH") else None,
            history_points=history, startup="available_history", target="next_sample", return_seasonality=None,
            refit="every_point", target_partition=partition)
        products[names[index]] = (frame, parts[0]["returns"]["schema"], "c03_ReturnModels",
                                  upstream, parameters, [("sample_id", "ascending")])
    options = dict(ARMA=dict(mean_order=(1, 1)), ARFIMA=dict(mean_order=(1, 1), fractional_memory=30),
                   HAR=dict(har_windows=(1, 5, 20)), MEM=dict(mem_order=(1, 1)))
    for index, (measure, model) in enumerate([(m, v) for m in ("rv", "bpv") for v in options], 6):
        frame = pd.concat([p["volatility"]["results"][(measure, model)] for p in parts],
                          ignore_index=True).sort_values("sample_id")
        parameters = dict(**options[model], model=model, observation=measure, transform="identity", seasonality=None,
            history_points=history, startup="available_history", require_nonnegative_prediction=True,
            refit="every_point", target_partition=partition)
        products[names[index]] = (frame, parts[0]["volatility"]["schema"], "c04_VolatilityModels",
                                  upstream, parameters, [("sample_id", "ascending")])
    _demo_save_node(code_root, output_dir, plan, rank, products)


In [ ]:
_demo_upload(demo_context)  # 只冻结同一份 full_inputs，不重新读取上游；不上传试错结果。


In [ ]:
demo_job = _demo_submit(demo_context)
print(f'{demo_job["name"]} | {demo_job.get("job_id", "本地成果复用")} | {demo_job.get("console_uri", "")}')
print("输出位置：", demo_job.get("actual_output_uri", demo_job.get("requested_output_uri", "")))


In [ ]:
_demo_wait(demo_context)  # 普通 Python 每 60 秒查同一 ID；明确失败后留证并结束，不重提。


In [ ]:
_demo_recover(demo_context)  # 接收所有节点、核对身份／完整覆盖、安装并复读；失败不重新计算。
print("完整 demo 数据已保存；展示可独立执行最后一格。")


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
candidate_root = Path.cwd().resolve()
while not (candidate_root / "config/settings.py").is_file():
    if candidate_root == candidate_root.parent:
        raise RuntimeError("找不到项目根目录；请在本项目内打开 Notebook")
    candidate_root = candidate_root.parent
sys.path[:0] = [str(candidate_root)]
from R04_Research.a01_Methods.b04_ReturnVolatilityModeling.c01_IntradaySeasonality import _demo_read_saved
demo_display_labels = {
    "sample_id": "sample_id（采样编号）", "sample_at": "sample_at（采样时刻）",
    "trading_date": "trading_date（交易日）", "sample_side": "sample_side（训练／测试侧）",
    "model": "model（模型）", "status": "status（状态）", "reason": "reason（原因）",
    "mean_forecast": "mean_forecast（下一步均值预测）",
    "variance_forecast": "variance_forecast（下一步条件方差）",
    "training_points": "training_points（训练采样点数）",
    "update_status": "update_status（观测更新状态）", "updated_mean_forecast": "updated_mean_forecast（更新后的下一步预测）",
    "parameters": "parameters（具名拟合系数）", "features": "features（HAR 特征）",
    "components": "components（MEM 成分）", "log_return": "log_return（区间对数收益）",
    "rv": "rv（区间平方收益贡献）", "bpv": "bpv（区间双幂变差贡献）",
    "log_return_status": "log_return_status（收益状态）", "rv_status": "rv_status（RV 状态）",
    "bpv_status": "bpv_status（BPV 状态）", "bar_at": "bar_at（分钟末时刻）",
    "rv_contribution": "rv_contribution（分钟平方收益贡献）",
    "bpv_contribution": "bpv_contribution（分钟双幂变差贡献）", "minute_slot": "minute_slot（分钟位置）",
    "seasonal_factor": "seasonal_factor（季节因子）", "causal_factor": "causal_factor（因果季节因子）",
    "is_initial_backfill": "is_initial_backfill（是否初始回填）"}
saved_demo_tables = _demo_read_saved(candidate_root, "models", "c03_ReturnModels")
for filename, saved_frame in saved_demo_tables.items():
    print(filename, f"；完整 {len(saved_frame):,} 行")
    selected_columns = [name for name in demo_display_labels if name in saved_frame.columns]
    display(saved_frame.loc[:, selected_columns].head(8).rename(columns=demo_display_labels))
    if "status" in saved_frame:
        display(saved_frame.groupby("status", dropna=False).size().rename("行数").rename_axis("status（状态）").to_frame())
# 本主题当前提供表格展示；若增加正式绘制函数，在此调用即可，不触发计算或保存试错图。
